### 3.1.1 Problem Definition

While BERT models achieve high accuracy in fake news detection, they operate as "black boxes." We know their predictions but not why. 

This helps us validate model decisions and understand what language features distinguish fake from real news.

### 3.1.2 Setup and Imports

In [1]:
import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from lime.lime_text import LimeTextExplainer
from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

# Device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Using device: {device}")
print(f"✅ Libraries loaded successfully!")

c:\Users\aweso\Documents\fakenews-analysis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Using device: cuda
✅ Libraries loaded successfully!


### 3.1.3 Model Configuration

In [2]:
# Define model paths
MODEL_1_PATH = '../../models/bert_fake_news_trained_v2'
MODEL_1_NAME = 'Combined Specialist'

MODEL_2_PATH = '../../models/bert_politifact_retrained'
MODEL_2_NAME = 'Political Specialist'

MODEL_3_PATH = '../../models/bert_gossip'
MODEL_3_NAME = 'Gossip Specialist'

print(f"📊 Model Configuration:")
print(f"  Model 1: {MODEL_1_NAME}")
print(f"  Model 2: {MODEL_2_NAME}")
print(f"  Model 3: {MODEL_3_NAME}")

📊 Model Configuration:
  Model 1: Combined Specialist
  Model 2: Political Specialist
  Model 3: Gossip Specialist


### 3.1.4 Load Models

In [3]:
# Load models
print(f"🔄 Loading models...")
model_1 = AutoModelForSequenceClassification.from_pretrained(MODEL_1_PATH)
tokenizer_1 = AutoTokenizer.from_pretrained(MODEL_1_PATH)
model_1.to(device)
model_1.eval()

model_2 = AutoModelForSequenceClassification.from_pretrained(MODEL_2_PATH)
tokenizer_2 = AutoTokenizer.from_pretrained(MODEL_2_PATH)
model_2.to(device)
model_2.eval()

model_3 = AutoModelForSequenceClassification.from_pretrained(MODEL_3_PATH)
tokenizer_3 = AutoTokenizer.from_pretrained(MODEL_3_PATH)
model_3.to(device)
model_3.eval()

print(f"✅ Models loaded successfully!")

🔄 Loading models...
✅ Models loaded successfully!
✅ Models loaded successfully!


### 3.1.5 Prediction and LIME Setup Functions

In [4]:
def predict_text(text, model, tokenizer, device):
    """Get prediction and probabilities for a text sample"""
    text = str(text).strip()
    inputs = tokenizer(text, padding=True, truncation=True, max_length=128, return_tensors='pt')
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1)
        prediction = torch.argmax(logits, dim=-1).item()
        fake_prob = probs[0][0].item()
        real_prob = probs[0][1].item()
    
    return prediction, {'fake': fake_prob, 'real': real_prob}, max(fake_prob, real_prob)

def create_predictor(model, tokenizer, device):
    """Create prediction function for LIME"""
    def predict_fn(texts):
        predictions = []
        for text in texts:
            text = str(text) if isinstance(text, (list, tuple)) else text
            pred, probs, conf = predict_text(text, model, tokenizer, device)
            if probs:
                predictions.append([probs['fake'], probs['real']])
            else:
                predictions.append([0.5, 0.5])
        return np.array(predictions)
    return predict_fn

# Setup LIME explainers
print("⚙️  Setting up LIME explainers...")
explainer_1 = LimeTextExplainer(class_names=['Fake News', 'Real News'])
explainer_2 = LimeTextExplainer(class_names=['Fake News', 'Real News'])
explainer_3 = LimeTextExplainer(class_names=['Fake News', 'Real News'])

predictor_1 = create_predictor(model_1, tokenizer_1, device)
predictor_2 = create_predictor(model_2, tokenizer_2, device)
predictor_3 = create_predictor(model_3, tokenizer_3, device)

print("✅ Predictors and explainers ready")

⚙️  Setting up LIME explainers...
✅ Predictors and explainers ready


### 3.1.6 Visualization and Analysis Function

In [5]:
def analyze_sentence_with_lime(sentence, model_1, tokenizer_1, model_2, tokenizer_2, model_3, tokenizer_3,
                                explainer_1, explainer_2, explainer_3, predictor_1, predictor_2, predictor_3,
                                device, top_n=12):
    """
    Comprehensive LIME analysis of a sentence with all three models.
    Shows predictions, word importance, and highlighted sentence.
    """
    
    print(f"\n{'='*100}")
    print(f"SENTENCE ANALYSIS")
    print(f"{'='*100}")
    print(f"\n📄 Input: {sentence[:100]}..." if len(sentence) > 100 else f"\n📄 Input: {sentence}")
    
    # Get predictions for all 3 models
    pred_1, probs_1, conf_1 = predict_text(sentence, model_1, tokenizer_1, device)
    pred_1_label = 'REAL' if pred_1 == 1 else 'FAKE'
    
    pred_2, probs_2, conf_2 = predict_text(sentence, model_2, tokenizer_2, device)
    pred_2_label = 'REAL' if pred_2 == 1 else 'FAKE'
    
    pred_3, probs_3, conf_3 = predict_text(sentence, model_3, tokenizer_3, device)
    pred_3_label = 'REAL' if pred_3 == 1 else 'FAKE'
    
    print(f"\n🤖 PREDICTIONS:")
    print(f"  {MODEL_1_NAME}: {pred_1_label} ({conf_1:.2%} confidence)")
    print(f"  {MODEL_2_NAME}: {pred_2_label} ({conf_2:.2%} confidence)")
    print(f"  {MODEL_3_NAME}: {pred_3_label} ({conf_3:.2%} confidence)")
    
    # Run LIME explanations for all 3 models
    print(f"\n⏳ Computing LIME explanations...")
    
    exp_1 = explainer_1.explain_instance(sentence, predictor_1, num_features=top_n, num_samples=1000)
    lime_weights_1 = exp_1.as_list()
    
    exp_2 = explainer_2.explain_instance(sentence, predictor_2, num_features=top_n, num_samples=1000)
    lime_weights_2 = exp_2.as_list()
    
    exp_3 = explainer_3.explain_instance(sentence, predictor_3, num_features=top_n, num_samples=1000)
    lime_weights_3 = exp_3.as_list()
    
    print(f"✅ LIME explanations complete!")
    
    # Create DataFrames for all 3 models
    df_1 = pd.DataFrame(lime_weights_1, columns=['word', 'contribution'])
    df_1['direction'] = df_1['contribution'].apply(lambda x: 'REAL' if x > 0 else 'FAKE')
    df_1['abs_contribution'] = abs(df_1['contribution'])
    df_1 = df_1.sort_values('abs_contribution', ascending=False)
    
    df_2 = pd.DataFrame(lime_weights_2, columns=['word', 'contribution'])
    df_2['direction'] = df_2['contribution'].apply(lambda x: 'REAL' if x > 0 else 'FAKE')
    df_2['abs_contribution'] = abs(df_2['contribution'])
    df_2 = df_2.sort_values('abs_contribution', ascending=False)
    
    df_3 = pd.DataFrame(lime_weights_3, columns=['word', 'contribution'])
    df_3['direction'] = df_3['contribution'].apply(lambda x: 'REAL' if x > 0 else 'FAKE')
    df_3['abs_contribution'] = abs(df_3['contribution'])
    df_3 = df_3.sort_values('abs_contribution', ascending=False)
    
    # === Sentence Highlighting for all 3 models ===
    print(f"\n📍 SENTENCE HIGHLIGHTING:")
    print(f"\n{MODEL_1_NAME}:")
    highlight_sentence(sentence, df_1)
    
    print(f"\n{MODEL_2_NAME}:")
    highlight_sentence(sentence, df_2)
    
    print(f"\n{MODEL_3_NAME}:")
    highlight_sentence(sentence, df_3)
    
    return {
        'sentence': sentence,
        'model_1_pred': pred_1_label,
        'model_1_conf': conf_1,
        'model_2_pred': pred_2_label,
        'model_2_conf': conf_2,
        'model_3_pred': pred_3_label,
        'model_3_conf': conf_3,
        'df_1': df_1,
        'df_2': df_2,
        'df_3': df_3
    }

def highlight_sentence(sentence, df_weights):
    """Create color-highlighted HTML version of sentence with smart word matching"""
    # Build contribution dictionary - LIME may return partial phrases
    # Keep both cleaned and original versions for matching
    lime_words = df_weights['word'].tolist()
    lime_words_lower = [w.lower().strip('.,!?;:') for w in lime_words]
    contrib_dict = dict(zip(lime_words_lower, df_weights['contribution'].tolist()))
    
    # Split sentence and process
    sentence_words = sentence.split()
    highlighted = []
    used_indices = set()
    
    i = 0
    while i < len(sentence_words):
        word = sentence_words[i]
        clean_word = word.lower().strip('.,!?;:')
        
        # Try multi-word phrases first (up to 4 words)
        found = False
        for phrase_len in [4, 3, 2]:
            if i + phrase_len <= len(sentence_words):
                # Build potential phrase
                phrase = ' '.join([w.lower().strip('.,!?;:') for w in sentence_words[i:i+phrase_len]])
                if phrase in contrib_dict:
                    # Found a match - highlight all words in phrase
                    score = contrib_dict[phrase]
                    
                    for j in range(i, i + phrase_len):
                        if score > 0:
                            color = f"rgba(0, 200, 0, 0.4)"  # Green for REAL
                        else:
                            color = f"rgba(255, 0, 0, 0.4)"  # Red for FAKE
                        
                        w = sentence_words[j]
                        highlighted.append(f"<span style='background-color:{color}; padding: 2px 4px; border-radius: 3px;'>{w}</span>")
                        used_indices.add(j)
                    i += phrase_len
                    found = True
                    break
        
        # If no multi-word phrase found, try single word
        if not found:
            if clean_word in contrib_dict:
                score = contrib_dict[clean_word]
                
                if score > 0:
                    color = f"rgba(0, 200, 0, 0.4)"  # Green for REAL
                else:
                    color = f"rgba(255, 0, 0, 0.4)"  # Red for FAKE
                highlighted.append(f"<span style='background-color:{color}; padding: 2px 4px; border-radius: 3px;'>{word}</span>")
                used_indices.add(i)
            else:
                # No match found, add unhighlighted
                highlighted.append(word)
            i += 1
    
    html_sentence = " ".join(highlighted)
    display(HTML(f"<p style='font-size:1.1em; line-height:1.8;'>{html_sentence}</p>"))

def analyze_semantic_patterns(df_1, df_2, df_3, model_1_name, model_2_name, model_3_name):
    """Analyze semantic categories in the top words"""
    # Define semantic categories
    emotional_words = {'demonic', 'infernal', 'wretched', 'malevolent', 'terrible', 'screeching', 'blasphemy', 'torment', 'vile', 'debauchery', 'shocked', 'anger', 'outrage'}
    sensational_words = {'unprecedented', 'shocking', 'milestone', 'historic', 'radical', 'innovative', 'extraordinary', 'remarkable', 'stunning', 'incredible'}
    vague_words = {'many', 'countless', 'some', 'several', 'strange', 'defy', 'unexplained', 'mysterious', 'unknown', 'alleged'}
    
    def count_category(words, category):
        return sum(1 for w in words if any(c in w.lower() for c in category))
    
    words_1 = df_1.head(15)['word'].str.lower().tolist()
    words_2 = df_2.head(15)['word'].str.lower().tolist()
    words_3 = df_3.head(15)['word'].str.lower().tolist()
    
    print(f"\n{model_1_name}:")
    print(f"  Emotional Language: {count_category(words_1, emotional_words)} words")
    print(f"  Sensational Markers: {count_category(words_1, sensational_words)} words")
    print(f"  Vague Language: {count_category(words_1, vague_words)} words")
    
    print(f"\n{model_2_name}:")
    print(f"  Emotional Language: {count_category(words_2, emotional_words)} words")
    print(f"  Sensational Markers: {count_category(words_2, sensational_words)} words")
    print(f"  Vague Language: {count_category(words_2, vague_words)} words")
    
    print(f"\n{model_3_name}:")
    print(f"  Emotional Language: {count_category(words_3, emotional_words)} words")
    print(f"  Sensational Markers: {count_category(words_3, sensational_words)} words")
    print(f"  Vague Language: {count_category(words_3, vague_words)} words")

### 3.1.7 Investigation Examples

In [6]:
# Test sentences - similar structure to TF-IDF investigation
test_sentences = [
    "Trump and Putin are meeting for a summit to discuss trade agreements and nuclear proliferation concerns",
    "SHOCKING: Trump's INCREDIBLE secret meeting with Putin REVEALED - You won't believe what they discussed!",
    "Federal officials announced new regulations for renewable energy development across multiple states"
]

# Analyze each sentence with all 3 models
results = []
for sentence in test_sentences:
    # Analyze with all 3 models (now included in the function)
    result = analyze_sentence_with_lime(
        sentence, 
        model_1, tokenizer_1, 
        model_2, tokenizer_2,
        model_3, tokenizer_3,
        explainer_1, explainer_2, explainer_3,
        predictor_1, predictor_2, predictor_3,
        device,
        top_n=12
    )
    
    results.append(result)


SENTENCE ANALYSIS

📄 Input: Trump and Putin are meeting for a summit to discuss trade agreements and nuclear proliferation conce...

🤖 PREDICTIONS:
  Combined Specialist: REAL (99.97% confidence)
  Political Specialist: REAL (88.93% confidence)
  Gossip Specialist: FAKE (96.90% confidence)

⏳ Computing LIME explanations...
✅ LIME explanations complete!

📍 SENTENCE HIGHLIGHTING:

Combined Specialist:
✅ LIME explanations complete!

📍 SENTENCE HIGHLIGHTING:

Combined Specialist:



Political Specialist:



Gossip Specialist:



SENTENCE ANALYSIS

📄 Input: SHOCKING: Trump's INCREDIBLE secret meeting with Putin REVEALED - You won't believe what they discus...

🤖 PREDICTIONS:
  Combined Specialist: FAKE (99.99% confidence)
  Political Specialist: FAKE (100.00% confidence)
  Gossip Specialist: FAKE (83.42% confidence)

⏳ Computing LIME explanations...
✅ LIME explanations complete!

📍 SENTENCE HIGHLIGHTING:

Combined Specialist:
✅ LIME explanations complete!

📍 SENTENCE HIGHLIGHTING:

Combined Specialist:



Political Specialist:



Gossip Specialist:



SENTENCE ANALYSIS

📄 Input: Federal officials announced new regulations for renewable energy development across multiple states

🤖 PREDICTIONS:
  Combined Specialist: REAL (99.89% confidence)
  Political Specialist: REAL (100.00% confidence)
  Gossip Specialist: FAKE (84.44% confidence)

⏳ Computing LIME explanations...
✅ LIME explanations complete!

📍 SENTENCE HIGHLIGHTING:

Combined Specialist:
✅ LIME explanations complete!

📍 SENTENCE HIGHLIGHTING:

Combined Specialist:



Political Specialist:



Gossip Specialist:
